In [ ]:
import sys
import os
# sys.path.append('./zennit-crp')  # Adjust to the correct relative path
# os.chdir("./zennit-crp")  # Change to the correct directory

import torch
# from torchvision.models.vgg import vgg16_bn, VGG16_BN_Weights
import torchvision.transforms as T
from PIL import Image
from zennit.canonizers import SequentialMergeBatchNorm
from zennit.composites import EpsilonPlusFlat

import torchvision
from crp.concepts import ChannelConcept
from crp.helper import get_layer_names
# from crp.attribution import CondAttribution
from crp.visualization import FeatureVisualization
from crp.image import imgify, plot_grid
from tutorials.VGG16_ImageNet.download_imagenet import download

import matplotlib.pyplot as plt
import numpy as np


In [ ]:
from zennit.composites import NameMapComposite
from zennit.core import Composite
from crp.hooks import MaskHook
from crp.concepts import Concept, ChannelConcept
from crp.graph import ModelGraph
from typing import Callable, List, Dict, Union, Tuple
import torch
import warnings
import numpy as np
import math
from tqdm import tqdm
from collections import namedtuple

attrResult = namedtuple("AttributionResults", "heatmap, activations, relevances, prediction")
attrGraphResult = namedtuple("AttributionGraphResults", "nodes, connections")


class CondAttribution:

    def __init__(self, model: torch.nn.Module, device: torch.device = None, overwrite_data_grad=True, no_param_grad=True) -> None:
        """
        This class contains the functionality to compute conditional attributions.

        Parameters:
        ----------
        model: torch.nn.Module
        device: torch.device
            specifies where the model and subsequent computation takes place.
        overwrite_data_grad: boolean
            If True, the .grad attribute of the 'data' argument is set to None before each __call__.
        no_param_grad: boolean
            If True, sets the requires_grad attribute of all model parameters to zero, to reduce the GPU memory footprint.
        """

        self.MODEL_OUTPUT_NAME = "y"

        self.device = next(model.parameters()).device if device is None else device
        self.model = model
        self.overwrite_data_grad = overwrite_data_grad

        if no_param_grad:
            self.model.requires_grad_(False)


    def backward(self, pred, grad_mask, partial_backward, layer_names, layer_out, generate=False):

        if partial_backward and len(layer_names) > 0:

            wrt_tensor, grad_tensors = pred, grad_mask.to(pred)

            for l_name in layer_names:

                inputs = layer_out[l_name]

                try:
                    grad = torch.autograd.grad(wrt_tensor, inputs=inputs, grad_outputs=grad_tensors, retain_graph=True)
                except RuntimeError as e:
                    if "allow_unused=True" not in str(e):
                        raise e
                    else:
                        raise RuntimeError(
                            "The layer names must be ordered according to their succession in the model if 'exclude_parallel'=True."
                            " Please make sure to start with the last and end with the first layer in each condition dict. In addition,"
                            " parallel layers can not be used in one condition.")

                # TODO: necessary?
                if grad is None:
                    raise RuntimeError(
                        "The layer names must be ordered according to their succession in the model if 'exclude_parallel'=True."
                        " Please make sure to start with the last and end with the first layer in each condition dict. In addition,"
                        " parallel layers can not be used in one condition.")

                wrt_tensor, grad_tensors = layer_out[l_name], grad

            torch.autograd.backward(wrt_tensor, grad_tensors, retain_graph=generate)

        else:

            torch.autograd.backward(pred, grad_mask.to(pred), retain_graph=generate)

    def relevance_init(self, prediction, target_list, init_rel):
        """

        Parameters:
        -----------
            prediction: torch.Tensor
                output of model forward pass
            target_list: list/numpy.ndarray or None
                list of all 'y' values of condition dictionaries. Indices are used to set the
                initial relevance to prediction values. If target_list is None and init_rel is None,
                relevance is initialized at all indices with prediction values. If start_layer is
                used, target_list is set to None.
            init_rel: torch.Tensor or None
                used to initialize relevance instead of prediction. If None, target_list is used.
                Please make sure to choose the right shape.
        """

        if callable(init_rel):
            output_selection = init_rel(prediction)
        elif isinstance(init_rel, torch.Tensor):
            output_selection = init_rel
        elif isinstance(init_rel, (int, np.integer)):
            output_selection = torch.full(prediction.shape, init_rel)
        else:
            output_selection = prediction

        if target_list:
            mask = torch.zeros_like(output_selection)
            print(f"target_list: {target_list}")
            print(f"output_selection: {output_selection}")
            print(f"prediction: {prediction}")
            print(f"prediction.shape: {prediction.shape}")
            print(f"output_selection.shape: {output_selection.shape}")
            print(f"init_rel: {init_rel}")
            for i, targets in enumerate(target_list):
                mask[i, targets] = output_selection[i, targets]
            output_selection = mask

        return output_selection

    def heatmap_modifier(self, data, on_device=None):

        heatmap = data.grad.detach()
        heatmap = heatmap.to(on_device) if on_device else heatmap
        return torch.sum(heatmap, dim=1)

    def broadcast(self, data, conditions) -> Tuple[torch.Tensor, Dict]:

        len_data, len_cond = len(data), len(conditions)

        if len_data == len_cond:
            data.retain_grad()
            return data, conditions

        if len_cond > 1:
            data = torch.repeat_interleave(data, len_cond, dim=0)
        if len_data > 1:
            conditions = conditions * len_data

        data.retain_grad()
        return data, conditions

    def _check_arguments(self, data, conditions, start_layer, exclude_parallel, init_rel):

        if not data.requires_grad:
            raise ValueError(
                "requires_grad attribute of 'data' must be True.")

        if self.overwrite_data_grad:
            data.grad = None
        elif data.grad is not None:
            warnings.warn("'data' already has a filled .grad attribute. Set to None if not intended or set 'overwrite_grad' to True.")

        distinct_cond = set()
        for cond in conditions:
            if self.MODEL_OUTPUT_NAME not in cond and start_layer is None and init_rel is None:
                raise ValueError(
                    f"Either {self.MODEL_OUTPUT_NAME} in 'conditions' or 'start_layer' or 'init_rel' must be defined.")

            if self.MODEL_OUTPUT_NAME in cond and start_layer is not None:
                warnings.warn(
                    f"You defined a condition for {self.MODEL_OUTPUT_NAME} that has no effect, since the 'start_layer' {start_layer}"
                    " is provided where the backward pass begins. If this behavior is not wished, remove 'start_layer'.")

            if exclude_parallel:

                if len(distinct_cond) == 0:
                    distinct_cond.update(cond.keys())
                elif distinct_cond ^ set(cond.keys()):
                    raise ValueError("If the 'exclude_parallel' flag is set to True, each condition dict must contain the"
                                     " same layer names. (This limitation does not apply to the __call__ method)")


    def _register_mask_fn(self, hook, mask_map, b_index, c_indices, l_name):

        if callable(mask_map):
            mask_fn = mask_map(b_index, c_indices, l_name)
        elif isinstance(mask_map, Dict):
            mask_fn = mask_map[l_name](b_index, c_indices, l_name)
        else:
            raise ValueError("<mask_map> must be a dictionary or callable function.")

        hook.fn_list.append(mask_fn)


    def __call__(
            self, data: torch.tensor, conditions: List[Dict[str, List]],
            composite: Composite = None, record_layer: List[str] = [],
            mask_map: Union[Callable, Dict[str, Callable]] = ChannelConcept.mask, start_layer: str = None, init_rel=None,
            on_device: str = None, exclude_parallel=True) -> attrResult:

        """
        Computes conditional attributions by masking the gradient flow of PyTorch (that is replaced by zennit with relevance values).
        The relevance distribution rules (as for LRP e.g.) are described in the zennit 'composite'. Relevance can be initialized at
        the model output or 'start_layer' with the 'init_rel' argument.
        How the relevances are masked is determined by the 'conditions' as well as the 'mask_map'. In addition, 'exclude_parallel'=True,
        restricts the PyTorch gradient flow so that it does not enter into parallel layers (shortcut connections) of the layers mentioned
        in the 'conditions' dictionary.
        The name of the model output is designated with self.MODEL_OUTPUT_NAME ('y' per default) and can be used inside 'conditions'.

        Parameters:
        -----------

        data: torch.Tensor
            Input sample for which a conditional heatmap is computed
        conditions: list of dict
            The key of a dict are string layer names and their value is a list of integers describing the concept (channel, neuron) index.
            In general, the values are passed to the 'mask_map' function as 'concept_ids' argument.
        composite: zennit Composite
            Object that describes how relevance is distributed. Should contain a suitable zennit Canonizer.
        mask_map: dict of callable or callable
            The keys of the dict are string layer names and the values functions that implement gradient masking. If no dict is used,
            all layers are masked according to the same function. 
            The 'conditions' values are passed into the function as 'concept_ids' argument.
        start_layer: (optional) str
            Layer name where to start the backward pass instead of starting at the model output. 
            If set, 'init_rel' modifies the tensor at 'start_layer' instead and a condition containing self.MODEL_OUTPUT_NAME is ignored.
        init_rel: (optional) torch.Tensor, int or callable
            Initializes the relevance distribution process as described in the LRP algorithm e.g. The callable must have the signature
            callable(activations).
            Per default, relevance is initialized with the logit activation before a non-linearity.
        on_device: (optional) str
            On which device (cpu, cuda) to save the heatmap, intermediate activations and relevances.
            Per default, everything is kept on the same device as the model parameters.
        exclude_parallel: boolean
            If set, the PyTorch gradient flow is restricted so that it does not enter into parallel layers (shortcut connections) 
            of the layers mentioned in the 'conditions' dictionary. Useful to get the sole contribution of a specific concept.

        Returns:
        --------
         
        attrResult: namedtuple object
            Contains the attributes 'heatmap', 'activations', 'relevances' and 'prediction'.
            'heatmap': torch.Tensor
                Output of the self.attribution_modifier method that defines how 'data'.grad is processed.
            'activations': dict of str and torch.Tensor
                The keys are the layer names and values are the activations
            'relevances': dict of str and torch.Tensor
                The keys are the layer names and values are the relevances
            'prediction': torch.Tensor
                The model prediction output. If 'start_layer' is set, 'prediction' is the layer activation.       
        """
        
        if exclude_parallel:
            return self._conditions_wrapper(data, conditions, composite, record_layer, mask_map, start_layer, init_rel, on_device, True)
        else:
            return self._attribute(data, conditions, composite, record_layer, mask_map, start_layer, init_rel, on_device, False)

    def _conditions_wrapper(self, *args):
        """
        Since 'exclude_parallel'=True requires that the condition set contains only the same layer names,
        the list is divided into distinct lists that all contain the same layer name.
        """

        data, conditions = args[:2]

        relevances, activations = {}, {}
        heatmap, prediction = None, None

        dist_conds = self._separate_conditions(conditions)

        for dist_layer in dist_conds:

            attr = self._attribute(data, dist_conds[dist_layer], *args[2:])

            for l_name in attr.relevances:
                if l_name not in relevances:
                    relevances[l_name] = attr.relevances[l_name]
                    activations[l_name] = attr.activations[l_name]
                else:
                    relevances[l_name] = torch.cat([relevances[l_name], attr.relevances[l_name]], dim=0)
                    activations[l_name] = torch.cat([activations[l_name], attr.activations[l_name]], dim=0)

            if heatmap is None:
                heatmap = attr.heatmap
                prediction = attr.prediction
            else:
                heatmap = torch.cat([heatmap, attr.heatmap], dim=0)
                prediction = torch.cat([prediction, attr.prediction], dim=0)

        return attrResult(heatmap, activations, relevances, prediction)

    def _separate_conditions(self, conditions):
        """
        Finds identical subsets of layer names inside 'conditions'
        """

        distinct_cond = dict()
        for cond in conditions:
            cond_set = frozenset(cond.keys())

            if cond_set in distinct_cond:
                distinct_cond[cond_set].append(cond)
            else:
                distinct_cond[cond_set] = [cond]

        return distinct_cond


    def _attribute(
            self, data: torch.tensor, conditions: List[Dict[str, List]],
            composite: Composite = None, record_layer: List[str] = [],
            mask_map: Union[Callable, Dict[str, Callable]] = ChannelConcept.mask, start_layer: str = None, init_rel=None,
            on_device: str = None, exclude_parallel=True) -> attrResult:
        """
        Computes the actual attributions as described in __call__ method docstring.
        exclude_parallel: boolean
            If set, all layer names in 'conditions' must be identical. This limitation does not apply to the __call__ method.
        """
        data, conditions = self.broadcast(data, conditions)

        self._check_arguments(data, conditions, start_layer, exclude_parallel, init_rel)

        hook_map, y_targets, cond_l_names = {}, [], []
        for i, cond in enumerate(conditions):
            for l_name, indices in cond.items():
                if l_name == self.MODEL_OUTPUT_NAME:
                    y_targets.append(indices)
                else:
                    if l_name not in hook_map:
                        hook_map[l_name] = MaskHook([])
                    self._register_mask_fn(hook_map[l_name], mask_map, i, indices, l_name)
                    if l_name not in cond_l_names:
                        cond_l_names.append(l_name)

        handles, layer_out = self._append_recording_layer_hooks(record_layer, start_layer, cond_l_names)

        name_map = [([name], hook) for name, hook in hook_map.items()]
        mask_composite = NameMapComposite(name_map)

        if composite is None:
            composite = Composite()

        with mask_composite.context(self.model), composite.context(self.model) as modified:

            if start_layer:
                _ = modified(data)
                pred = layer_out[start_layer]
                grad_mask = self.relevance_init(pred.detach().clone(), None, init_rel)
                if start_layer in cond_l_names:
                    cond_l_names.remove(start_layer)
                self.backward(pred, grad_mask, exclude_parallel, cond_l_names, layer_out)

            else:
                pred = modified(data)
                grad_mask = self.relevance_init(pred.detach().clone(), y_targets, init_rel)
                self.backward(pred, grad_mask, exclude_parallel, cond_l_names, layer_out)

            attribution = self.heatmap_modifier(data, on_device)
            activations, relevances = {}, {}
            if len(layer_out) > 0:
                activations, relevances = self._collect_hook_activation_relevance(layer_out, on_device)
            [h.remove() for h in handles]

        return attrResult(attribution, activations, relevances, pred)

    def generate(
            self, data: torch.tensor, conditions: List[Dict[str, List]],
            composite: Composite = None, record_layer: List[str] = [],
            mask_map: Union[Callable, Dict[str, Callable]] = ChannelConcept.mask, start_layer: str = None, init_rel=None,
            batch_size=10, on_device=None, exclude_parallel=True, verbose=True) -> attrResult:
        """
        Computes several conditional attributions for single data point by broadcasting 'data' to length 'batch_size' and
        iterating through the 'conditions' list with stepsize 'batch_size'. The model forward pass is performed only once and 
        the backward graph kept in memory in order to double the performance.
        Please refer to the docstring of the __call__ method.

        batch_size: int
            batch size of each forward and backward pass
        exclude_parallel: boolean
            If set, all layer names in 'conditions' must be identical. This limitation does not apply to the __call__ method.
        verbose: boolean
            If set, a progressbar is displayed.
        """

        self._check_arguments(data, conditions, start_layer, exclude_parallel, init_rel)

        # register on all layers in layer_map an empty hook
        hook_map, cond_l_names = {}, []
        for cond in conditions:
            for l_name in cond.keys():
                if l_name not in hook_map:
                    hook_map[l_name] = MaskHook([])
                if l_name != self.MODEL_OUTPUT_NAME and l_name not in cond_l_names:
                    cond_l_names.append(l_name)

        handles, layer_out = self._append_recording_layer_hooks(record_layer, start_layer, cond_l_names)

        name_map = [([name], hook) for name, hook in hook_map.items()]
        mask_composite = NameMapComposite(name_map)

        if composite is None:
            composite = Composite()

        cond_length = len(conditions)
        if cond_length > batch_size:
            batches = math.ceil(cond_length / batch_size)
        else:
            batches = 1
            batch_size = cond_length

        data_batch = torch.repeat_interleave(data, batch_size, dim=0)
        data_batch.grad = None
        data_batch.retain_grad()
        retain_graph = True

        with mask_composite.context(self.model), composite.context(self.model) as modified:

            if start_layer:
                _ = modified(data_batch)
                pred = layer_out[start_layer]
                if start_layer in cond_l_names:
                    cond_l_names.remove(start_layer)

            else:
                pred = modified(data_batch)

            if verbose:
                pbar = tqdm(total=batches, dynamic_ncols=True)

            for b in range(batches):

                if verbose:
                    pbar.update(1)

                cond_batch = conditions[b * batch_size: (b + 1) * batch_size]

                y_targets = []
                for i, cond in enumerate(cond_batch):
                    for l_name, indices in cond.items():
                        if l_name == self.MODEL_OUTPUT_NAME:
                            y_targets.append(indices)
                        else:
                            self._register_mask_fn(hook_map[l_name], mask_map, i, indices, l_name)

                if b == batches-1:
                    # last batch may have len(y_targets) != batch_size. Padded part is ignored later.
                    # and backward graph is freed with retain_graph=False
                    if not start_layer:
                        y_targets.extend([y_targets[0] for i in range(batch_size-len(y_targets))])
                    batch_size = len(cond_batch)
                    retain_graph = False

                grad_mask = self.relevance_init(pred.detach().clone(), y_targets, init_rel)
                self.backward(pred, grad_mask, exclude_parallel, cond_l_names, layer_out, retain_graph)

                heatmap = self.heatmap_modifier(data_batch)
                activations, relevances = {}, {}
                if len(layer_out) > 0:
                    activations, relevances = self._collect_hook_activation_relevance(
                        layer_out, on_device, batch_size)

                yield attrResult(heatmap[:batch_size], activations, relevances, pred[:batch_size])

                self._reset_gradients(data_batch)
                [hook.fn_list.clear() for hook in hook_map.values()]

        [h.remove() for h in handles]

        if verbose:
            pbar.close()

    @staticmethod
    def _generate_hook(layer_name, layer_out):
        def get_tensor_hook(module, input, output):
            layer_out[layer_name] = output
            output.retain_grad()

        return get_tensor_hook

    def _append_recording_layer_hooks(self, record_l_names: list, start_layer, cond_l_names):
        """
        applies a forward hook to all layers in record_l_names, start_layer and cond_l_names to record 
        the activations and relevances
        """

        handles = []
        layer_out = {}
        record_l_names = record_l_names.copy()

        for l_name in cond_l_names:
            if l_name not in record_l_names:
                record_l_names.append(l_name)

        if start_layer is not None and start_layer not in record_l_names:
            record_l_names.append(start_layer)

        for name, layer in self.model.named_modules():

            if name == self.MODEL_OUTPUT_NAME:
                raise ValueError(
                    "No layer name should match the constant for the identifier of the model output."
                    "Please change the layer name or the OUTPUT_NAME constant of the object."
                    "Note, that the condition set then references to the output with OUTPUT_NAME and no longer 'y'.")

            if name in record_l_names:
                h = layer.register_forward_hook(self._generate_hook(name, layer_out))
                handles.append(h)
                record_l_names.remove(name)

        if start_layer in record_l_names:
            raise KeyError(f"<start_layer> {start_layer} not found in model.")
        if len(record_l_names) > 0:
            warnings.warn(
                f"Some layer names not found in model: {record_l_names}.")

        return handles, layer_out

    def _collect_hook_activation_relevance(self, layer_out, on_device=None, length=None):
        """

        Parameters:
        ----------
            layer_out: dict
                contains the intermediate layer outputs
            on_device: str
                copy layer_out on cpu or cuda device
            length: int
                copy only first length elements of layer_out. Used for uneven batch sizes.
        """

        relevances = {}
        activations = {}
        for name in layer_out:
            act = layer_out[name].detach()[:length]
            activations[name] = act.to(on_device) if on_device else act
            activations[name].requires_grad = False

            if layer_out[name].grad is None:
                rel = torch.zeros_like(activations[name], requires_grad=False)[:length]
                relevances[name] = rel.to(on_device) if on_device else rel
            else:
                rel = layer_out[name].grad.detach()[:length]
                relevances[name] = rel.to(on_device) if on_device else rel
                relevances[name].requires_grad = False
                layer_out[name].grad = None

        return activations, relevances

    def _reset_gradients(self, data):
        """
        custom zero_grad() function
        """

        for p in self.model.parameters():
            p.grad = None

        data.grad = None

In [ ]:
from typing import List, Union, Dict, Tuple, Callable
import warnings
import torch
import numpy as np
import math
from collections.abc import Iterable
import concurrent.futures
import functools
import inspect
from tqdm import tqdm
from zennit.composites import NameMapComposite, Composite
# from crp.attribution import CondAttribution
from crp.maximization import Maximization
from crp.concepts import ChannelConcept, Concept
from crp.statistics import Statistics
from crp.hooks import FeatVisHook
from crp.helper import load_maximization, load_statistics, load_stat_targets
from crp.image import vis_img_heatmap, vis_opaque_img
from crp.cache import Cache


class FeatureVisualization:

    def __init__(
            self, attribution: CondAttribution, dataset, layer_map: Dict[str, Concept], preprocess_fn: Callable=None,
            max_target="sum", abs_norm=True, path="FeatureVisualization", device=None, cache: Cache=None):

        self.dataset = dataset
        self.layer_map = layer_map
        self.preprocess_fn = preprocess_fn

        self.attribution = attribution

        self.device = attribution.device if device is None else device

        self.RelMax = Maximization("relevance", max_target, abs_norm, path)
        self.ActMax = Maximization("activation", max_target, abs_norm, path)

        self.RelStats = Statistics("relevance", max_target, abs_norm, path)
        self.ActStats = Statistics("activation", max_target, abs_norm, path)

        self.Cache = cache

    def preprocess_data(self, data):

        if callable(self.preprocess_fn):
            return self.preprocess_fn(data)
        else:
            return data

    def get_data_sample(self, index, preprocessing=True) -> Tuple[torch.Tensor, int]:
        """
        returns a data sample from dataset at index.

        Parameter:
            index: integer
            preprocessing: boolean.
                If True, return the sample after preprocessing. If False, return the sample for plotting.
        """

        data, target = self.dataset[index]
        data = data.to(self.device).unsqueeze(0)
        if preprocessing:
            data = self.preprocess_data(data)
        
        data.requires_grad = True
        return data, target

    def multitarget_to_single(self, multi_target):

        raise NotImplementedError

    def run(self, composite: Composite, data_start, data_end, batch_size=32, checkpoint=500, on_device=None):

        print("Running Analysis...")
        saved_checkpoints = self.run_distributed(composite, data_start, data_end, batch_size, checkpoint, on_device)

        print("Collecting results...")
        saved_files = self.collect_results(saved_checkpoints)

        return saved_files

    def run_distributed(self, composite: Composite, data_start, data_end, batch_size=16, checkpoint=500, on_device=None):
        """
        max batch_size = max(multi_targets) * data_batch
        data_end: exclusively counted
        """

        self.saved_checkpoints = {"r_max": [], "a_max": [], "r_stats": [], "a_stats": []}
        last_checkpoint = 0

        n_samples = data_end - data_start
        samples = np.arange(start=data_start, stop=data_end)

        if n_samples > batch_size:
            batches = math.ceil(n_samples / batch_size)
        else:
            batches = 1
            batch_size = n_samples

        # feature visualization is performed inside forward and backward hook of layers
        name_map, dict_inputs = [], {}
        for l_name, concept in self.layer_map.items():
            hook = FeatVisHook(self, concept, l_name, dict_inputs, on_device)
            name_map.append(([l_name], hook))
        fv_composite = NameMapComposite(name_map)

        if composite:
            composite.register(self.attribution.model)
        fv_composite.register(self.attribution.model)

        pbar = tqdm(total=batches, dynamic_ncols=True)

        for b in range(batches):

            pbar.update(1)

            samples_batch = samples[b * batch_size: (b + 1) * batch_size]
            data_batch, targets_samples = self.get_data_concurrently(samples_batch, preprocessing=True)

            targets_samples = np.array(targets_samples)  # numpy operation needed

            # convert multi target to single target if user defined the method
            data_broadcast, targets, sample_indices = [], [], []
            try:
                for i_t, target in enumerate(targets_samples):
                    single_targets = self.multitarget_to_single(target)
                    for st in single_targets:
                        targets.append(st)
                        data_broadcast.append(data_batch[i_t])
                        sample_indices.append(samples_batch[i_t])
                if len(data_broadcast) == 0:
                    continue
                # TODO: test stack
                data_broadcast = torch.stack(data_broadcast, dim=0)
                sample_indices = np.array(sample_indices)
                targets = np.array(targets)

            except NotImplementedError:
                data_broadcast, targets, sample_indices = data_batch, targets_samples, samples_batch

            conditions = [{self.attribution.MODEL_OUTPUT_NAME: [t]} for t in targets]
            # dict_inputs is linked to FeatHooks
            dict_inputs["sample_indices"] = sample_indices
            dict_inputs["targets"] = targets

            # composites are already registered before
            self.attribution(data_broadcast, conditions, None, exclude_parallel=False)

            if b % checkpoint == checkpoint - 1:
                self._save_results((last_checkpoint, sample_indices[-1] + 1))
                last_checkpoint = sample_indices[-1] + 1

        # TODO: what happens if result arrays are empty?
        self._save_results((last_checkpoint, sample_indices[-1] + 1))

        if composite:
            composite.remove()
        fv_composite.remove()

        pbar.close()

        return self.saved_checkpoints

    @torch.no_grad()
    def analyze_relevance(self, rel, layer_name, concept, data_indices, targets):
        """
        Finds input samples that maximally activate each neuron in a layer and most relevant samples
        """
        d_c_sorted, rel_c_sorted, rf_c_sorted, t_c_sorted = self.RelMax.analyze_layer(
            rel, concept, layer_name, data_indices, targets)

        self.RelStats.analyze_layer(d_c_sorted, rel_c_sorted, rf_c_sorted, t_c_sorted, layer_name)

    @torch.no_grad()
    def analyze_activation(self, act, layer_name, concept, data_indices, targets):
        """
        Finds input samples that maximally activate each neuron in a layer and most relevant samples
        """

        # activation analysis once per sample if multi target dataset
        unique_indices = np.unique(data_indices, return_index=True)[1]
        data_indices = data_indices[unique_indices]
        act = act[unique_indices]
        targets = targets[unique_indices]

        d_c_sorted, act_c_sorted, rf_c_sorted, t_c_sorted = self.ActMax.analyze_layer(
            act, concept, layer_name, data_indices, targets)

        self.ActStats.analyze_layer(d_c_sorted, act_c_sorted, rf_c_sorted, t_c_sorted, layer_name)

    def _save_results(self, d_index=None):

        self.saved_checkpoints["r_max"].extend(self.RelMax._save_results(d_index))
        self.saved_checkpoints["a_max"].extend(self.ActMax._save_results(d_index))
        self.saved_checkpoints["r_stats"].extend(self.RelStats._save_results(d_index))
        self.saved_checkpoints["a_stats"].extend(self.ActStats._save_results(d_index))

    def collect_results(self, checkpoints: Dict[str, List[str]], d_index: Tuple[int, int] = None):

        saved_files = {}

        saved_files["r_max"] = self.RelMax.collect_results(checkpoints["r_max"], d_index)
        saved_files["a_max"] = self.ActMax.collect_results(checkpoints["a_max"], d_index)
        saved_files["r_stats"] = self.RelStats.collect_results(checkpoints["r_stats"], d_index)
        saved_files["a_stats"] = self.ActStats.collect_results(checkpoints["a_stats"], d_index)

        return saved_files

    def get_data_concurrently(self, indices: Union[List, np.ndarray, torch.tensor], preprocessing=False):

        if len(indices) == 1:
            data, label = self.get_data_sample(indices[0], preprocessing)
            return data, label

        threads = []
        data_returned = []
        labels_returned = []
        with concurrent.futures.ThreadPoolExecutor() as executor:
            for index in indices:
                future = executor.submit(self.get_data_sample, index, preprocessing)
                threads.append(future)

        for t in threads:
            single_data = t.result()[0]
            single_label = t.result()[1]
            data_returned.append(single_data)
            labels_returned.append(single_label)

        data_returned = torch.cat(data_returned, dim=0)
        return data_returned, labels_returned


    def cache_reference(func):
        """
        Decorator for get_max_reference and get_stats_reference. If a crp.cache object is supplied to the FeatureVisualization object,
        reference samples are cached i.e. saved after computing a visualization with a 'plot_fn' (argument of get_max_reference) or
        loaded from the disk if available.
        """
        @functools.wraps(func)
        def wrapper(self, *args, **kwargs):
            """
            Parameters:
            -----------
            overwrite: boolean
                If set to True, already computed reference samples are computed again (overwritten).
            """
            
            overwrite = kwargs.pop("overwrite", False)
            args_f = inspect.getcallargs(func, self, *args, **kwargs)
            plot_fn = args_f["plot_fn"]

            if self.Cache is None or plot_fn is None:
                return func(**args_f)

            r_range, mode, l_name, rf, composite = args_f["r_range"], args_f["mode"], args_f["layer_name"], args_f["rf"], args_f["composite"]
            f_name, plot_name = func.__name__, plot_fn.__name__
            if f_name == "get_max_reference":
                indices = args_f["concept_ids"]
            else:
                indices = [f'{args_f["concept_id"]}:{i}' for i in args_f["targets"]]

            if overwrite:
                not_found = {id: r_range for id in indices}
                ref_c = {}
            else:
                ref_c, not_found = self.Cache.load(indices, l_name, mode, r_range, composite, rf, f_name, plot_name)

            if len(not_found):
                
                for id in not_found:
                    
                    args_f["r_range"] = not_found[id]

                    if f_name == "get_max_reference":
                        args_f["concept_ids"]  = id
                        ref_c_left = func(**args_f)
                    elif f_name == "get_stats_reference":
                        args_f["targets"] = int(id.split(":")[-1])
                        ref_c_left = func(**args_f)
                    else:
                        raise ValueError("Only the methods 'get_max_reference' and 'get_stats_reference' can be decorated.")

                    self.Cache.save(ref_c_left, l_name, mode, not_found[id], composite, rf, f_name, plot_name)

                    ref_c = self.Cache.extend_dict(ref_c, ref_c_left)

            return ref_c

        return wrapper

    @cache_reference
    def get_max_reference(
            self, concept_ids: Union[int,list], layer_name: str, mode="relevance", r_range: Tuple[int, int] = (0, 8), composite: Composite=None,
            rf=False, plot_fn=vis_img_heatmap, batch_size=32)-> Dict:
        """
        Retreive reference samples for a list of concepts in a layer. Relevance and Activation Maximization
        are availble if FeatureVisualization was computed for the mode. In addition, conditional heatmaps can be computed on reference samples.
        If the crp.concept class (supplied to the FeatureVisualization layer_map) implements masking for a single neuron in the 'mask_rf' method, 
        the reference samples and heatmaps can be cropped using the receptive field of the most relevant or active neuron.

        Parameters:
        ----------
        concept_ids: int or list
        layer_name: str
        mode: "relevance" or "activation"
            Relevance or Activation Maximization 
        r_range: Tuple(int, int)
            Range of N-top reference samples. For example, (3, 7) corresponds to the Top-3 to -6 samples.
            Argument must be a closed set i.e. second element of tuple > first element.
        composite: zennit.composites or None
            If set, compute conditional heatmaps on reference samples. `composite` is used for the CondAttribution object.
        rf: boolean
            If True, compute the CRP heatmap for the most relevant/most activating neuron only to restrict the conditonal heatmap
            on the receptive field.
        plot_fn: callable function with signature (samples: torch.Tensor, heatmaps: torch.Tensor, rf: boolean) or None
            Draws reference images. The function receives as input the samples used for computing heatmaps before preprocessing 
            with self.preprocess_data and the final heatmaps after computation. In addition, the boolean flag 'rf' is passed to it.
            The return value of the function should correspond to the Cache supplied to the FeatureVisualization object (if available).
            If None, the raw tensors are returned.
        batch_size: int
            If heatmap is True, describes maximal batch size of samples to compute for conditional heatmaps.

        Returns:
        -------
        ref_c: dictionary.
            Key values correspond to channel index and values are reference samples. The values depend on the implementation of
            the 'plot_fn'.
        """

        ref_c = {}
        if not isinstance(concept_ids, Iterable):
            concept_ids = [concept_ids]
        if mode == "relevance":
            d_c_sorted, _, rf_c_sorted = load_maximization(self.RelMax.PATH, layer_name)
        elif mode == "activation":
            d_c_sorted, _, rf_c_sorted = load_maximization(self.ActMax.PATH, layer_name)
        else:
            raise ValueError("`mode` must be `relevance` or `activation`")

        if rf and not composite:
            warnings.warn("The receptive field is only computed, if you fill the 'composite' argument with a zennit Composite.")

        for c_id in concept_ids:

            d_indices = d_c_sorted[r_range[0]:r_range[1], c_id]
            n_indices = rf_c_sorted[r_range[0]:r_range[1], c_id]

            ref_c[c_id] = self._load_ref_and_attribution(d_indices, c_id, n_indices, layer_name, composite, rf, plot_fn, batch_size)

        return ref_c

    @cache_reference
    def get_stats_reference(self, concept_id: int, layer_name: str, targets: Union[int, list], mode="relevance", r_range: Tuple[int, int] = (0, 8),
            composite=None, rf=False, plot_fn=vis_img_heatmap, batch_size=32):
        """
        Retreive reference samples for a single concept in a layer wrt. different explanation targets i.e. returns the reference samples
        that are computed by self.compute_stats. Relevance and Activation are availble if FeatureVisualization was computed for the statitics mode. 
        In addition, conditional heatmaps can be computed on reference samples. If the crp.concept class (supplied to the FeatureVisualization layer_map) 
        implements masking for a single neuron in the 'mask_rf' method, the reference samples and heatmaps can be cropped using the receptive field of 
        the most relevant or active neuron.

        Parameters:
        ----------
        concept_ids: int or list
        layer_name: str
        mode: "relevance" or "activation"
            Relevance or Activation Maximization 
        r_range: Tuple(int, int)
            Range of N-top reference samples. For example, (3, 7) corresponds to the Top-3 to -6 samples.
            Argument must be a closed set i.e. second element of tuple > first element.
        composite: zennit.composites or None
            If set, compute conditional heatmaps on reference samples. `composite` is used for the CondAttribution object.
        rf: boolean
            If True, compute the CRP heatmap for the most relevant/most activating neuron only to restrict the conditonal heatmap
            on the receptive field.
        plot_fn: callable function with signature (samples: torch.Tensor, heatmaps: torch.Tensor, rf: boolean)
            Draws reference images. The function receives as input the samples used for computing heatmaps before preprocessing 
            with self.preprocess and the final heatmaps after computation. In addition, the boolean flag 'rf' is passed to it.
            The return value of the function should correspond to the Cache supplied to the FeatureVisualization object (if available).
            If None, the raw tensors are returned.
        batch_size: int
            If heatmap is True, describes maximal batch size of samples to compute for conditional heatmaps.

        Returns:
        -------
        ref_t: dictionary.
            Key values correspond to target indices and values are reference samples. The values depend on the implementation of
            the 'plot_fn'.
        """
            
        
        ref_t = {}
        if not isinstance(targets, Iterable):
            targets = [targets]
        if mode == "relevance":
            path = self.RelStats.PATH
        elif mode == "activation":
            path = self.ActStats.PATH 
        else:
            raise ValueError("`mode` must be `relevance` or `activation`")
        
        if rf and not composite:
            warnings.warn("The receptive field is only computed, if you fill the 'composite' argument with a zennit Composite.")

        for t in targets:
            
            d_c_sorted, _, rf_c_sorted = load_statistics(path, layer_name, t)
            d_indices = d_c_sorted[r_range[0]:r_range[1], concept_id]
            n_indices = rf_c_sorted[r_range[0]:r_range[1], concept_id]

            ref_t[f"{concept_id}:{t}"] = self._load_ref_and_attribution(d_indices, concept_id, n_indices, layer_name, composite, rf, plot_fn, batch_size)

        return ref_t

    def _load_ref_and_attribution(self, d_indices, c_id, n_indices, layer_name, composite, rf, plot_fn, batch_size):

        data_batch, _ = self.get_data_concurrently(d_indices, preprocessing=False)

        if composite:
            data_p = self.preprocess_data(data_batch)
            heatmaps = self._attribution_on_reference(data_p, c_id, layer_name, composite, rf, n_indices, batch_size)

            if callable(plot_fn):
                return plot_fn(data_batch.detach(), heatmaps.detach(), rf)
            else:
                return data_batch.detach().cpu(), heatmaps.detach().cpu()

        else:
            return data_batch.detach().cpu()

    def _attribution_on_reference(self, data, concept_id: int, layer_name: str, composite, rf=False, neuron_ids: list=[], batch_size=32):

        n_samples = len(data)
        if n_samples > batch_size:
            batches = math.ceil(n_samples / batch_size)
        else:
            batches = 1
            batch_size = n_samples

        if rf and (len(neuron_ids) != n_samples):
            raise ValueError("length of 'neuron_ids' must be equal to the length of 'data'")

        heatmaps = []
        for b in range(batches):
            data_batch = data[b * batch_size: (b + 1) * batch_size].detach().requires_grad_()
            
            if rf:
                batch_neuron_ids = neuron_ids[b * batch_size: (b + 1) * batch_size]
                conditions = [{layer_name: {concept_id: n_index}} for n_index in batch_neuron_ids]
                attr = self.attribution(data_batch, conditions, composite, mask_map=ChannelConcept.mask_rf, start_layer=layer_name, on_device=self.device, 
                    exclude_parallel=False)
            else:
                conditions = [{layer_name: [concept_id]}] 
                # initialize relevance with activation before non-linearity (could be changed in a future release)
                attr = self.attribution(data_batch, conditions, composite, start_layer=layer_name, on_device=self.device, exclude_parallel=False)

            heatmaps.append(attr.heatmap)

        return torch.cat(heatmaps, dim=0)

    def compute_stats(self, concept_id, layer_name: str, mode="relevance", top_N=5, mean_N=10, norm=False) -> Tuple[list, list]:
        """
        Computes statistics about the targets i.e. classes that are most relevant or most activating for the concept with index 'concept_id'
        in layer 'layer_name'. Statistics must be computed before utilizing this method.

        Parameters:
        -----------
        concept_id: int
            Index of concept
        layer_name: str
        mode: str, 'relevance' or 'activation'
        top_N: int
            Returns the 'top_N' classes that most activate or are most relevant for the concept.
        mean_N: int
            Computes the importance of each target using the 'mean_N' top reference images for each target.
        norm: boolean
            If True, returns the mean relevance for each target normed.

        Returns:
        --------
        sorted_t, sorted_val as tuple
        sorted_t: list of most relevant targets
        sorted_val: list of respective mean relevance/activation values for each target 
        """

        if mode == "relevance":
            path = self.RelStats.PATH
        elif mode == "activation":
            path = self.ActStats.PATH 
        else:
            raise ValueError("`mode` must be `relevance` or `activation`")
        
        targets = load_stat_targets(path)

        rel_target = torch.zeros(len(targets))
        for i, t in enumerate(targets):
            _, rel_c_sorted, _ = load_statistics(path, layer_name, t)
            rel_target[i] = float(rel_c_sorted[:mean_N, concept_id].mean())
        
        args = torch.argsort(rel_target, descending=True)[:top_N]

        sorted_t = targets[args]
        sorted_val = rel_target[args]

        if norm:
            sorted_val = sorted_val / sorted_val[0]
        
        return sorted_t, sorted_val

    def _save_precomputed(self, s_tensor, h_tensor, index, plot_list, layer_name, mode, r_range, composite, rf, f_name):

        for plot_fn in plot_list:
            ref = {index: plot_fn(s_tensor, h_tensor, rf)}
            self.Cache.save(ref, layer_name, mode, r_range, composite, rf, f_name, plot_fn.__name__)

    def precompute_ref(self, layer_c_ind:Dict[str, List], composite: Composite, rf=True, stats=False, top_N=4, mean_N=10, mode="relevance", r_range: Tuple[int, int] = (0, 8), plot_list=[vis_opaque_img], batch_size=32):
        """
        Precomputes and saves all reference samples resulting from 'self.get_ref_samples' and 'self.get_stats_reference' for concepts supplied in 'layer_c_ind'.

        Parameters:
        -----------
        layer_c_ind: dict with str keys and list values
            Keys correspond to layer names and values to a list of all concept indices
        stats: boolean
            If True, precomputes reference samples of 'self.get_stats_reference'. Otherwise, only samples of 'self.get_ref_samples' are computed.
        plot_list: list of callable functions
            Functions to plot and save the images. The signature should correspond to the 'plot_fn' of 'get_max_reference'.

        REMAINING PARAMETERS: correspond to 'self.get_ref_samples' and 'self.get_stats_reference'
        """


        if self.Cache is None:
            raise ValueError("You must supply a crp.Cache object to the 'FeatureVisualization' class to precompute reference images!")
        
        if composite is None:
            raise ValueError("You must supply a zennit.Composite object to precompute reference images!")

        for l_name in layer_c_ind:

            c_indices = layer_c_ind[l_name]
            print("Layer:", l_name)
            pbar = tqdm(total=len(c_indices), dynamic_ncols=True)

            for c_id in c_indices:

                s_tensor, h_tensor = self.get_max_reference(c_id, l_name, mode, r_range, composite, rf, None, batch_size)[c_id]

                self._save_precomputed(s_tensor, h_tensor, c_id, plot_list, l_name, mode, r_range, composite, rf, "get_max_reference")
                                   
                if stats:
                    targets, _ = self.compute_stats(c_id, l_name, mode, top_N, mean_N)
                    for t in targets:
                        stat_index = f"{c_id}:{t}"
                        s_tensor, h_tensor = self.get_stats_reference(c_id, l_name, t, mode, r_range, composite, rf, None, batch_size)[stat_index]
                        self._save_precomputed(s_tensor, h_tensor, stat_index, plot_list, l_name, mode, r_range, composite, rf, "get_stats_reference")

                pbar.update(1)

            pbar.close()

       

In [ ]:
class ChannelConceptMax(ChannelConcept):
    def attribute(self, relevance, mask=None, layer_name: str | None = None, abs_norm=True):
        if isinstance(mask, torch.Tensor):
            relevance = relevance * mask

        rel_l = torch.max(relevance.view(*relevance.shape[:2], -1), dim=-1).values

        if abs_norm:
            rel_l = rel_l / (torch.abs(rel_l).sum(-1).view(-1, 1) + 1e-10)

        return rel_l

In [ ]:
from pathlib import Path
# import mlflow
from src.modeling.decode_head.binary_classifier import BinaryClassifier
import albumentations



device = "cuda:0" if torch.cuda.is_available() else "cpu"

# mlflow.set_tracking_uri('http://mlflow.rationai-mlflow:5000/')
# logged_model = 'runs:/c38a483305574912a2b3f15bc64a2284/model/MLFlowModelCheckpoint/vgg16_prostate_best'

# # Load model as a PyFuncModel.
# model = mlflow.pyfunc.load_model(logged_model)
# model = vgg16_bn(weights=VGG16_BN_Weights.IMAGENET1K_V1)

model_state_dict = torch.load("state_dict_version1_tl_best.pth", map_location=device)
model_features = torchvision.models.vgg16().features
model_features.load_state_dict(model_state_dict['features'])
model_decoder = BinaryClassifier(in_features=512)
model_decoder.load_state_dict(model_state_dict['decode_head'])

class VGG16_Rationai(torch.nn.Module):
    def __init__(self, features, decode_head):
        super(VGG16_Rationai, self).__init__()
        self.features = features
        self.decode_head = decode_head

    def forward(self, x):
        x = self.features(x)
        x = self.decode_head(x)

        x = torch.cat((x, 1 - x), dim=0)  # return both classes for binary classification
        return x

model = VGG16_Rationai(
    features=model_features,
    decode_head=model_decoder
)
model = model.to(device)

model.eval()

layer_names = get_layer_names(model, [torch.nn.Conv2d, torch.nn.Linear])

attribution = CondAttribution(model)

canonizers = [SequentialMergeBatchNorm()]
composite = EpsilonPlusFlat(canonizers)

# Original code for VGG16_bn
# # separate normalization from resizing for plotting purposes later
# transform = T.Compose([T.Resize(256), T.CenterCrop(224), T.ToTensor()])
# preprocessing =  T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

# transform_norm = T.Compose([
#     transform,
#     preprocessing
# ])

preprocessing = T.Compose([])
# transform = T.Normalize(
#         mean=[230.6765, 187.0875, 222.7069],
#         std=[23.1706, 45.7364, 22.656]
#     )

mean = [230.6765, 187.0875, 222.7069]
std = [23.1706, 45.7364, 22.656]

inverse_transform = T.Compose([
    lambda x: x * torch.tensor(std).view(3, 1, 1),
    lambda x: x + torch.tensor(mean).view(3, 1, 1),
])
transform = T.Compose([])
transform_norm = T.Compose([
    preprocessing,
    transform
])



# data_path = Path("tutorials/ImageNet_data")

# if len(list(data_path.iterdir())) > 0:
#     print(f"Using existing data at {data_path}")
# else:  # download ImageNet validation set
#     download(data_path)

# # apply no normalization here!
# imagenet_data = torchvision.datasets.ImageNet(data_path, transform=transform, split="val")

In [ ]:
import hydra
import mlflow
from omegaconf import DictConfig, OmegaConf
import os

os.environ["HYDRA_FULL_ERROR"] = "1"  # to see full error messages in case of errors

# load the default config from the config directory
with hydra.initialize(version_base=None, config_path="conf"):
        cfg = hydra.compose(config_name="crp-prov-gigapath", overrides=[])

mlflow.set_tracking_uri("http://mlflow.rationai-mlflow:5000/")

# instantiate the data module using the default config
data_module = hydra.utils.instantiate(
    cfg.data,
    _recursive_=True,
    _convert_="partial",
)

# uris = ["mlflow-artifacts:/65/7382f77c8a404d479d5b070146d62f86/artifacts/Prostate - test"]
# dataset = ProstateCancer(
#     uris=uris,
#     transforms=albumentations.Normalize(
#         mean=[230.6765, 187.0875, 222.7069],
#         std=[23.1706, 45.7364, 22.656]
#     )
# )
# data_module = DataModule(
#     dataset=dataset,
#     batch_size=4,
#     num_workers=2,
# )



In [ ]:
dl = data_module.test_dataloader()

class ProstateWrapper(torch.utils.data.Dataset):
    """This wrapper just removes metadata from the (X, Y, metadata) triple that the RationAI data module returns.
    It returns only the (X, Y) pair.
    """
    def __init__(self, dataset):
        self.dataset = dataset

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        x, y, _ = self.dataset[idx]
        # make Y one-hot encoded (instead of a single number)
        # x = x.to(device)
        # y = torch.concat((y, 1-y)) # convert to binary classification format
        return x, int(y)  # convert to binary classification format

print(len(dl))
print(dl.dataset[166][0].dtype, type(dl.dataset[166][1]))
print(dl.dataset[166][0].shape, dl.dataset[166][1])
# print(type(dl.dataset[166][1]))


dataset = ProstateWrapper(dl.dataset)
print(dataset[166][0].dtype, type(dataset[166][1]))
print(dataset[166][0].shape, dataset[166][1])

res_ = model(dataset[166][0].unsqueeze(0).to(device))
print(res_.shape, res_.dtype)


In [ ]:
display(layer_names)

plt.imshow(inverse_transform(dataset[166][0]).cpu().numpy().transpose(1, 2, 0).astype(np.uint8))
plt.show()



In [ ]:
# ORIGINAL CODE for VGG16_BN
# def get_ith_last_feature_layer(i):
#     # get the ith last feature layer
#     return layer_names[-4 - i]

def get_ith_last_feature_layer(i):
    # get the ith last feature layer
    return layer_names[-2 - i]

In [ ]:
# ORIGINAL CODE for VGG16_BN
# assert get_ith_last_feature_layer(0) == "features.40"
# assert get_ith_last_feature_layer(1) == "features.37"

assert get_ith_last_feature_layer(0) == "features.28", "Expected features.28, got " + get_ith_last_feature_layer(0)
assert get_ith_last_feature_layer(1) == "features.26", "Expected features.26, got " + get_ith_last_feature_layer(1)

In [ ]:
# ORIGINAL CODE for VGG16_BN
# fv_path = "tutorials/VGG16_ImageNet"

# def feature_visualization(concept, layer_names):
#     fv = FeatureVisualization(attribution, imagenet_data, { name: concept for name in layer_names }, preprocess_fn=preprocessing, path=fv_path)
#     return fv

fv_path = "tutorials/VGG16_Prostate"
def feature_visualization(concept, layer_names):
    fv = FeatureVisualization(attribution, dataset, { name: concept for name in layer_names }, preprocess_fn=preprocessing, path=fv_path)
    
    return fv


In [ ]:
# def get_ids(sample, prob=0.05):
#     y = model(sample)
#     # print(y, y.shape)
#     probs = softmax(y).squeeze()  # shape: (num_classes,)
#     mask = probs >= prob
#     selected_ids = mask.nonzero(as_tuple=True)[0]
#     selected_probs = probs[selected_ids]

#     # Sort by probability in descending order
#     sorted_indices = torch.argsort(selected_probs, descending=True)
#     sorted_ids = selected_ids[sorted_indices]
#     sorted_probs = selected_probs[sorted_indices]

#     return sorted_ids.tolist(), sorted_probs.tolist()

def get_ids(sample, prob=0.5):
    # y = model(sample.unsqueeze().to(device))  # shape: (1, num_classes, h, w)
    probs = model(sample).squeeze()
    print(f"Probabilities: {probs}, {probs.shape}")  # shape: (num_classes,)
    mask = probs >= prob
    selected_ids = mask.nonzero(as_tuple=True)[0]
    print(f"Selected IDs: {selected_ids}, Probabilities: {probs}, {type(probs)}")
    selected_probs = probs[selected_ids]

    # Sort by probability in descending order
    sorted_indices = torch.argsort(selected_probs, descending=True)
    sorted_ids = selected_ids[sorted_indices]
    sorted_probs = selected_probs[sorted_indices]

    return sorted_ids.tolist(), sorted_probs.tolist()


In [ ]:
# ORIGINAL CODE for ImageNet dataset
# def get_label(id):
#     return imagenet_data.classes[id]

def get_label(id):
    return "positive" if id == 1 else "negative" if id == 0 else "heck! there's a problem somewhere"

In [ ]:
# ORIGINAL CODE for ImageNet dataset
# def get_image(path):
#     image = Image.open(path)
#     sample = transform_norm(image).unsqueeze(0).to(device)

#     # zennit requires gradients
#     sample.requires_grad = True
    
#     return image, sample

from PIL import Image


def get_image(sample):
    # sample is already a tensor, no need to load from path
    image = Image.fromarray(inverse_transform(sample).cpu().numpy().transpose(1, 2, 0).astype(np.uint8))
    sample = sample.to(device)
    sample.requires_grad = True  # zennit requires gradients
    return image, sample

In [ ]:
def get_conditions(y, channels_sequence):
    channels_names = [get_ith_last_feature_layer(i) for i in range(len(channels_sequence))]
    channels = {name: [id]  for name, id in zip(channels_names, channels_sequence)}
    conditions = {'y' : [y], **channels}

    return conditions

In [ ]:
def get_top_concepts(sample, y, channels_sequence, concept_attribution, num_of_concepts):
    attr = attribution(sample, [get_conditions(y, channels_sequence)], composite, record_layer=layer_names)

    rel_c = concept_attribution.attribute(attr.relevances[get_ith_last_feature_layer(len(channels_sequence))], abs_norm=True)

    rel_values_tensor, concept_ids_tensor = torch.topk(rel_c[0], num_of_concepts)
    concept_ids = [int(id) for id in concept_ids_tensor]
    rel_values = [float(value) * 100 for value in rel_values_tensor]

    return concept_ids, rel_values

In [ ]:
def show_top_concepts(sample, concept_ids,  y, channels_sequence):
    conditions =  get_conditions(y, channels_sequence)
    last_layer = get_ith_last_feature_layer(len(channels_sequence))
    new_conditions = [{**conditions, last_layer: [id]} for id in concept_ids]
    heatmap, _, _, _ = attribution(sample, new_conditions, composite)
    display(imgify(heatmap, symmetric=True, grid=(1, len(concept_ids))))

In [ ]:
def show_top_representatives(feature_visualization, concept_ids, layer, num_of_representatives):
    ref_c = feature_visualization.get_max_reference(concept_ids, layer, "relevance", (0, num_of_representatives), composite=composite, plot_fn=None)

    for id, images in zip(concept_ids, ref_c.values()):
        print(f"Concept {id}")
        display(imgify(images[0], grid=(1, num_of_representatives)))
        display(imgify(images[1], symmetric=True, grid=(1, num_of_representatives)))

In [ ]:
def run(sample, y, channels_sequence, concept_attribution, feature_visualization, num_of_concepts, num_of_representatives):
    print(f"Predicted label {get_label(y)} (id: {y})")
    layer = get_ith_last_feature_layer(len(channels_sequence))
    print(f'Showing layer {layer}')

    concept_ids, rel_values = get_top_concepts(sample, y, channels_sequence, concept_attribution, num_of_concepts)
    
    print(f"Top {num_of_concepts} concepts are {concept_ids} with relevance {rel_values}")

    show_top_concepts(sample, concept_ids, y, channels_sequence)

    show_top_representatives(feature_visualization, concept_ids, layer, num_of_representatives)



In [ ]:
print(layer_names)

In [ ]:
# Original code for IMAGE_NET dataset
# feature_visualization(ChannelConcept(), [get_ith_last_feature_layer(i) for i in [2, 3, 4]]).run(composite,  0, len(imagenet_data), 32, 100)
import logging

logging.basicConfig(level=logging.INFO)

# feature_visualization(ChannelConcept(), [get_ith_last_feature_layer(i) for i in [0, 1]]).run(composite=composite,  data_start=0, data_end=104, batch_size=8, checkpoint=10)


In [ ]:
concept_sum = ChannelConcept()
# concept_max = ChannelConceptMax()
fv_sum = feature_visualization(concept_sum, layer_names)
# fv_max = feature_visualization(concept_max, layer_names)

In [ ]:
# Original code for IMAGE_NET dataset
# image, sample = get_image("tutorials/images/lizard.jpg")
image, sample = get_image(dl.dataset[166][0])

print(sample.shape, sample.dtype)

display(image)

ids, probs = get_ids(sample)

display(ids, probs)

run(sample, ids[0], [], concept_sum, fv_sum, num_of_concepts=10, num_of_representatives=8)

In [ ]:
image, sample = get_image("tutorials/images/lizard.jpg")

display(image)

ids, probs = get_ids(sample)

display(ids, probs)

run(sample, ids[0], [], concept_max, fv_max, num_of_concepts=10, num_of_representatives=8)

In [ ]:
image, sample = get_image("tutorials/ImageNet_data/val/n07697313/ILSVRC2012_val_00008200.JPEG")
display(image)

ids, probs = get_ids(sample)

display(ids, probs)

run(sample, ids[0], [], concept_sum, fv_sum, num_of_concepts=10, num_of_representatives=8)

In [ ]:
image, sample = get_image("tutorials/ImageNet_data/val/n13133613/ILSVRC2012_val_00000412.JPEG")
run(sample, ids[0], [], concept_sum, fv_sum, num_of_concepts=10, num_of_representatives=8)

In [ ]:
image, sample = get_image("tutorials/ImageNet_data/val/n01775062/ILSVRC2012_val_00005290.JPEG")
display(image)

ids, probs = get_ids(sample)

display(ids, probs)

run(sample, ids[0], [], concept_sum, fv_sum, num_of_concepts=10, num_of_representatives=8)

In [ ]:
image, sample = get_image("tutorials/ImageNet_data/val/n01775062/ILSVRC2012_val_00040214.JPEG")
display(image)

ids, probs = get_ids(sample)

display(ids, probs)

run(sample, ids[0], [], concept_sum, fv_sum, num_of_concepts=10, num_of_representatives=8)

In [ ]:
image, sample = get_image("tutorials/ImageNet_data/val/n01775062/ILSVRC2012_val_00040214.JPEG")
display(image)

ids, probs = get_ids(sample)

# display(ids, probs)

run(sample, ids[0], [71], concept_sum, fv_sum, num_of_concepts=10, num_of_representatives=8)

In [ ]:
image, sample = get_image("tutorials/images/lizard.jpg")

display(image)

ids, probs = get_ids(sample)

display(ids, probs)

run(sample, ids[0], [], concept_sum, fv_sum, num_of_concepts=10, num_of_representatives=8)

In [ ]:
image, sample = get_image("tutorials/ImageNet_data/val/n13052670/ILSVRC2012_val_00008275.JPEG")
run(image, sample, concept_sum, fv_sum)